# Optimise SplitGaussianWaveform on E/I SNN

Optimises the parameters of a `SplitGaussianWaveform` to match a target
population PSTH produced by the `RandomEINetwork` model (O'Rawe et al. 2023
membrane and opsin parameters).

**Strategy**
1. Define a *ground-truth* waveform with known parameters.
2. Run the SNN with that waveform to obtain the target PSTH.
3. Optimise a waveform starting from a different initial guess, using PSTH
   MSE as the objective.
4. Verify the optimised waveform recovers the target PSTH.

> **Note — simulation cost.**  Each objective evaluation runs a full Brian2
> simulation.  A reduced network (2 000 E + 500 I neurons, 200 ms pre/post)
> is used here for speed.  Swap in the full config for production runs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from designer_waveform.waveforms import SplitGaussianWaveform
from designer_waveform.models import (
    RandomEINetwork,
    RandomEINetworkConfig,
    NetworkParams,
    SimulationParams,
)

## 1. Build model

Load the O'Rawe parameter config and override the network size and timing
for faster iteration during optimisation.

In [ ]:
CONFIG_PATH = Path("..") / "configs" / "random_ei_orawe_params.toml"

cfg = RandomEINetworkConfig.from_toml(CONFIG_PATH)

# Reduce network size and simulation duration for faster optimisation.
# Swap back to the full config values for production runs.
cfg.network = NetworkParams(N_exc=2000, N_inh=500, p_conn=0.02)
cfg.simulation = SimulationParams(
    dt_ms=0.1,
    t_pre_ms=200.0,   # ms — enough for network to reach steady state
    t_stim_ms=200.0,  # ms — stimulation window
    t_post_ms=100.0,  # ms
    psth_bin_ms=5.0,
    seed=20250319,
)

model = RandomEINetwork(cfg)
print(f"Opsin distribution — mean: {model._stim_dist_pA.mean():.1f} pA, "
      f"frac zero: {(model._stim_dist_pA == 0).mean():.3f}")

## 2. Generate target PSTH

Run the network once with a known ground-truth waveform (asymmetric
Gaussian: fast rise, slow decay) to obtain the target PSTH.

In [ ]:
true_waveform = SplitGaussianWaveform(
    amplitude=1.0,
    mu=40.0,        # peak at 40 ms into the stim window
    sigma_rise=15.0,
    sigma_fall=50.0,
    baseline=0.0,
)

print("Running target simulation...")
target_result = model.run(true_waveform)
target_psth   = target_result["psth_exc"]
t_psth_ms     = target_result["t_psth_ms"]
t_stim_ms     = target_result["t_stim_ms"]
print("Done.")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].plot(t_stim_ms, true_waveform(t_stim_ms), color="steelblue", lw=2)
axes[0].set_xlabel("time in stim window (ms)")
axes[0].set_ylabel("waveform amplitude (a.u.)")
axes[0].set_title("Ground-truth waveform")

axes[1].bar(t_psth_ms, target_psth, width=cfg.simulation.psth_bin_ms * 0.9,
            color="steelblue", alpha=0.8)
axes[1].set_xlabel("time in stim window (ms)")
axes[1].set_ylabel("spikes / neuron / bin")
axes[1].set_title("Target PSTH (excitatory population)")

plt.tight_layout()
plt.show()

## 3. Define objective function

In [ ]:
PENALTY_WEIGHT = 1e4  # soft non-negativity penalty weight

def psth_mse(waveform: SplitGaussianWaveform) -> float:
    """MSE between simulated and target PSTH, with a soft non-negativity
    penalty on the waveform to discourage negative stimulus values."""
    result = model.run(waveform)
    psth   = result["psth_exc"]

    loss = float(np.mean((psth - target_psth) ** 2))

    # Penalise negative waveform values (stimulus should be non-negative)
    y = waveform(t_stim_ms)
    negativity = float(np.mean(np.clip(-y, 0, None) ** 2))
    loss += PENALTY_WEIGHT * negativity

    return loss

## 4. Run optimisation

Start from a symmetric Gaussian with a different peak time and width,
then let Nelder-Mead adjust amplitude, mu, sigma_rise, sigma_fall, and
baseline to minimise the PSTH MSE.

> Each iteration runs one full SNN simulation (~seconds on a laptop with
> the reduced network).  Reduce `maxiter` for a quick test.

In [ ]:
init_waveform = SplitGaussianWaveform(
    amplitude=0.5,
    mu=100.0,
    sigma_rise=40.0,
    sigma_fall=40.0,
    baseline=0.0,
)

result_waveform, opt = init_waveform.optimise(
    psth_mse,
    method="Nelder-Mead",
    verbose=True,
    log_every=10,
    options={"maxiter": 300, "xatol": 1e-4, "fatol": 1e-7},
)

print(f"\nMin waveform value : {result_waveform(t_stim_ms).min():.4f}  (should be ≥ 0)")
print(f"True waveform      : {true_waveform}")
print(f"Result waveform    : {result_waveform}")

## 5. Compare results

In [ ]:
opt_result   = model.run(result_waveform)
opt_psth     = opt_result["psth_exc"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Waveforms
ax = axes[0]
ax.plot(t_stim_ms, true_waveform(t_stim_ms),   color="steelblue", lw=2, label="True")
ax.plot(t_stim_ms, init_waveform(t_stim_ms),   color="grey",       lw=1.5, ls="--", label="Initial")
ax.plot(t_stim_ms, result_waveform(t_stim_ms), color="tomato",     lw=2, label="Optimised")
ax.set_xlabel("time in stim window (ms)")
ax.set_ylabel("waveform amplitude (a.u.)")
ax.set_title("Waveforms")
ax.legend()

# Target PSTH
bin_w = cfg.simulation.psth_bin_ms * 0.9
ax = axes[1]
ax.bar(t_psth_ms, target_psth, width=bin_w, color="steelblue", alpha=0.7, label="Target")
ax.bar(t_psth_ms, opt_psth,    width=bin_w * 0.6, color="tomato", alpha=0.8, label="Optimised")
ax.set_xlabel("time in stim window (ms)")
ax.set_ylabel("spikes / neuron / bin")
ax.set_title("PSTH comparison")
ax.legend()

# Residuals
ax = axes[2]
ax.bar(t_psth_ms, opt_psth - target_psth, width=bin_w, color="purple", alpha=0.7)
ax.axhline(0, color="k", lw=0.8, ls="--")
ax.set_xlabel("time in stim window (ms)")
ax.set_ylabel("residual (spikes / neuron / bin)")
ax.set_title("Optimised − Target")

plt.tight_layout()
plt.show()

mse_init = np.mean((model.run(init_waveform)["psth_exc"] - target_psth) ** 2)
mse_opt  = np.mean((opt_psth - target_psth) ** 2)
print(f"MSE — initial: {mse_init:.4e}   optimised: {mse_opt:.4e}   "
      f"improvement: {mse_init / mse_opt:.1f}×")